In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import ConnectionConfig as cc
cc.setupEnvironment()

In [3]:
spark = cc.startLocalCluster("DIM_SLOT") # default 4
spark.getActiveSession()

In [5]:
cc.set_connectionProfile("VeloDB")

df_operational_slots = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "locks") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "lockid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

df_operational_stations = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "stationid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()



In [7]:
df_operational_slots.createOrReplaceTempView("dimSlots")
df_operational_stations.createOrReplaceTempView("dimStations")

In [8]:
#TEMP VIEW
import uuid

# df_locks.createOrReplaceTempView("dimLocks")
# df_stations.createOrReplaceTempView("stations")

#TRANSFORM
# todo  niet uuid id gebruiken
df_dim_locks = spark.sql("select uuid() as lockSK, l.lockId, l.stationlocknr, s.stationnr, s.type, s.street, s.number, s.zipcode, s.district, s.labelid FROM dimSlots as l LEFT JOIN dimStations s on l.stationId = s.stationId")

df_geen_slot = spark.createDataFrame(
    [(str(uuid.uuid4()), "geen lockId", "geen stationlocknr", "geen station", "geen type", "geen straat", "geen nr", "geen zipcd", "geen district", "geen labelid")],
    ["lockSK", "lockId", "stationlocknr", "stationnr", "type", "street", "number", "zipcode", "district", "labelid"]
)

df_dim_locks = df_dim_locks.union(df_geen_slot)

df_dim_locks.createOrReplaceTempView("dimSlots")

spark.sql("select * from dimSlots").show()

+--------------------+------+-------------+---------+------------+--------------------+------+-------+---------+-------+
|              lockSK|lockId|stationlocknr|stationnr|        type|              street|number|zipcode| district|labelid|
+--------------------+------+-------------+---------+------------+--------------------+------+-------+---------+-------+
|fb864d12-c7d1-4f8...|     1|            1|      026|DUBBELZIJDIG|         Meir (2000)|    84|   2000|ANTWERPEN|   NULL|
|03b9e492-6f7e-47d...|     2|            2|      026|DUBBELZIJDIG|         Meir (2000)|    84|   2000|ANTWERPEN|   NULL|
|f8efd740-acb7-45f...|     3|            3|      026|DUBBELZIJDIG|         Meir (2000)|    84|   2000|ANTWERPEN|   NULL|
|8b814cba-e911-453...|     4|            4|      026|DUBBELZIJDIG|         Meir (2000)|    84|   2000|ANTWERPEN|   NULL|
|2fa8c404-615c-438...|     5|            5|      026|DUBBELZIJDIG|         Meir (2000)|    84|   2000|ANTWERPEN|   NULL|
|df824f12-7dda-4af...|     6|   

In [9]:
delta_table_path = "spark-warehouse/dimSlot"
# dimDate.write.format("delta").mode("overwrite").save(delta_table_path)

df_dim_locks.write.format("delta").mode("overwrite").save(delta_table_path)


In [14]:
df_dim_locks.show()
df_dim_locks.write.format("delta").mode("overwrite").saveAsTable("dimLocks")


+--------------------+------+-------------+---------+------------+--------------------+------+-------+---------+-------+
|              lockSK|lockId|stationlocknr|stationnr|        type|              street|number|zipcode| district|labelid|
+--------------------+------+-------------+---------+------------+--------------------+------+-------+---------+-------+
|13d4d128-e918-434...|     1|            1|      026|DUBBELZIJDIG|         Meir (2000)|    84|   2000|ANTWERPEN|   NULL|
|22f26dda-7853-401...|     2|            2|      026|DUBBELZIJDIG|         Meir (2000)|    84|   2000|ANTWERPEN|   NULL|
|e0156a3e-94d4-458...|     3|            3|      026|DUBBELZIJDIG|         Meir (2000)|    84|   2000|ANTWERPEN|   NULL|
|39e46aeb-1b9e-4df...|     4|            4|      026|DUBBELZIJDIG|         Meir (2000)|    84|   2000|ANTWERPEN|   NULL|
|7f3dc41a-7f58-41b...|     5|            5|      026|DUBBELZIJDIG|         Meir (2000)|    84|   2000|ANTWERPEN|   NULL|
|bb2da3d3-1a88-4b2...|     6|   